# Uji beda dua model: apakah model B benar-benar lebih baik dari A?
**Kapan dipakai:** sebelum memilih model akhir atau saat juri bertanya "selisih 0,005 itu
nyata atau kebetulan?". Bekerja di atas dua file OOF (`oof_<A>.npy`, `oof_<B>.npy`) yang baris dan
kolomnya sama, tanpa melatih ulang.

1. **Bootstrap berpasangan**: baris diambil ulang acak (dengan pengembalian) `N_BOOT` kali, skor A
   dan B dihitung di baris yang sama → selang 95% selisih dan peluang B > A.
2. **Per lipatan**: selisih skor di tiap lipatan CV (lipatan sama dengan notebook OOF).
3. **McNemar** (khusus kelas): baris yang benar di satu model saja, uji binomial.

Aturan praktis: selang 95% tidak memuat 0 dan B menang di ≥ 4 dari 5 lipatan → pilih B.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.stats import binomtest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/uji_beda_model"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba

TARGET = "rating"                    # GANTI saat lomba
MODEL_A = "demo_kata"                # GANTI: nama OOF pembanding (lama)
MODEL_B = "demo_kata_huruf"          # GANTI: nama OOF penantang (baru)
TASK = "kelas"                       # GANTI: "angka" untuk regresi (OOF n × 1)
N_BOOT = 1000
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    if TASK == "kelas":
        return f1_score(truth, pred.argmax(axis=1), average="macro")     # GANTI: metrik lomba
    return -np.sqrt(mean_squared_error(truth, pred[:, 0]))               # -RMSE: besar = baik


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca target + lipatan (sama dengan notebook OOF: Stratified untuk kelas, KFold untuk angka)

In [ ]:
train = pd.read_csv(TRAIN_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
if TASK == "kelas":
    CLASSES, truth = np.unique(train[TARGET].astype(str), return_inverse=True)
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
else:
    truth = train[TARGET].astype(float).to_numpy()
    splitter = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(splitter.split(train, truth))
print("baris:", len(truth))

## OOF demo (HAPUS saat lomba)
Dua model cepat: TF-IDF kata saja (A) vs kata + huruf (B), ditulis ke `OUTPUT_FOLDER`.
Saat lomba OOF sudah ada di `OOF_FOLDER`.

In [ ]:
def demo_oof(analyzers):
    text = train["title"].fillna("") + " . " + train["review_text"].fillna("")
    blocks = []
    for analyzer in analyzers:
        vectorizer = TfidfVectorizer(analyzer=analyzer, sublinear_tf=True, min_df=2,
                                     max_features=100000)
        blocks.append(vectorizer.fit_transform(text))
    matrix = hstack(blocks).tocsr()
    oof = np.zeros((len(train), len(CLASSES)))
    for idx_fit, idx_val in folds:
        fitted = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
        fitted.fit(matrix[idx_fit], truth[idx_fit])
        oof[np.ix_(idx_val, fitted.classes_)] = fitted.predict_proba(matrix[idx_val])
    return oof


OOF_FOLDER = OUTPUT_FOLDER          # demo tidak mengotori folder OOF asli
np.save(f"{OOF_FOLDER}/oof_{MODEL_A}.npy", demo_oof(["word"]))
np.save(f"{OOF_FOLDER}/oof_{MODEL_B}.npy", demo_oof(["word", "char_wb"]))
print("OOF demo ditulis")

## Muat dua OOF

In [ ]:
oof_a = np.load(f"{OOF_FOLDER}/oof_{MODEL_A}.npy").reshape(len(truth), -1)
oof_b = np.load(f"{OOF_FOLDER}/oof_{MODEL_B}.npy").reshape(len(truth), -1)
score_a = metric(truth, oof_a)
score_b = metric(truth, oof_b)
print(f"A {MODEL_A}: {score_a:.4f} | B {MODEL_B}: {score_b:.4f}",
      f"| selisih B−A: {score_b - score_a:+.4f}")

## 1. Bootstrap berpasangan

In [ ]:
rng = np.random.default_rng(SEED)
gaps = np.zeros(N_BOOT)
for number in range(N_BOOT):
    rows = rng.integers(0, len(truth), len(truth))
    gaps[number] = metric(truth[rows], oof_b[rows]) - metric(truth[rows], oof_a[rows])
low, high = np.quantile(gaps, [0.025, 0.975])
share_b = (gaps > 0).mean()
print(f"selang 95% selisih: [{low:+.4f}, {high:+.4f}] | peluang B > A: {share_b:.3f}")

## 2. Per lipatan + 3. McNemar

In [ ]:
rows = []
for fold, (_, idx_val) in enumerate(folds):
    fold_a = metric(truth[idx_val], oof_a[idx_val])
    fold_b = metric(truth[idx_val], oof_b[idx_val])
    rows.append({"lipatan": fold, "A": round(fold_a, 4), "B": round(fold_b, 4),
                 "B−A": round(fold_b - fold_a, 4)})
per_fold = pd.DataFrame(rows)
wins = int((per_fold["B−A"] > 0).sum())
print(per_fold.to_string(index=False), f"\nB menang di {wins} dari {N_FOLDS} lipatan")
mcnemar_p = np.nan
if TASK == "kelas":
    right_a = oof_a.argmax(axis=1) == truth
    right_b = oof_b.argmax(axis=1) == truth
    only_b = int((right_b & ~right_a).sum())
    only_a = int((right_a & ~right_b).sum())
    mcnemar_p = binomtest(only_b, only_b + only_a, 0.5).pvalue
    print(f"McNemar: benar di B saja {only_b}, di A saja {only_a}, p = {mcnemar_p:.4g}")

## Ringkasan + gambar (bahan slide)

In [ ]:
verdict = "belum pasti, pilih yang lebih sederhana/cepat"
if low > 0 and wins >= N_FOLDS - 1:
    verdict = "B lebih baik"
if high < 0 and wins <= 1:
    verdict = "A lebih baik"
summary = pd.DataFrame([{"A": MODEL_A, "B": MODEL_B, "skor A": round(score_a, 4),
                         "skor B": round(score_b, 4), "selang 95% bawah": round(low, 4),
                         "selang 95% atas": round(high, 4), "peluang B > A": round(share_b, 3),
                         "B menang lipatan": wins, "p McNemar": mcnemar_p, "kesimpulan": verdict}])
summary.to_csv(f"{OUTPUT_FOLDER}/uji_beda_{MODEL_A}_vs_{MODEL_B}.csv", index=False)
figure, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(gaps, bins=40, color="#0072B2", alpha=0.8)
ax.axvline(0, color="#1a1a1a", linewidth=1)
ax.axvline(low, color="#D55E00", linestyle="--", linewidth=1)
ax.axvline(high, color="#D55E00", linestyle="--", linewidth=1)
ax.set_xlabel(f"selisih skor B − A ({MODEL_B} − {MODEL_A})")
ax.set_ylabel("jumlah bootstrap")
figure.savefig(f"{OUTPUT_FOLDER}/beda_bootstrap.png", dpi=200, bbox_inches="tight")
plt.show()
print("kesimpulan:", verdict)